# ProbeForge 0.2 — Colab A100 pilot
Choose an A100 runtime. This notebook installs the pinned training stack; restart the runtime after installation if prompted. Upload the release ZIP. No model training has been performed in this notebook. Save checkpoints to mounted Drive so runtime expiration does not erase them.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
uploaded = files.upload()
import zipfile
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    z.extractall('/content')
%cd /content/probeforge
%pip install -e '.[train,quant,energy]'


## Check environment and download
Model files require disk space. If imports fail after dependency installation, restart the runtime and return to this directory.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
print(torch.cuda.get_device_name(0))
!contrastforge doctor
!contrastforge download --model Qwen/Qwen3-8B --out models/Qwen3-8B


## Data pilot
60 training worlds produce 1,600 training prompts. Use a fresh directory when changing the dataset. Natural-language templates need bilingual review before research claims.

In [ ]:
!contrastforge generate --out data/contrast --groups 60
!contrastforge preflight --inputs data/contrast/train.jsonl data/contrast/validation.jsonl data/contrast/test.jsonl --tokenizer models/Qwen3-8B --max-length 2048


In [ ]:
!contrastforge collect --inputs data/contrast/train.jsonl data/contrast/validation.jsonl --out data/student_base.jsonl --model models/Qwen3-8B --quantize --budgets 512


In [ ]:
import subprocess, sys
for arm in ['translated', 'random', 'mined', 'no_counterfactual']:
    subprocess.run([sys.executable, '-m', 'contrastforge.cli', 'prepare', '--inputs', 'data/contrast/train.jsonl', 'data/contrast/validation.jsonl', '--rollouts', 'data/student_base.jsonl', '--strategy', arm, '--tokenizer', 'models/Qwen3-8B', '--token-budget', '2000000', '--out', f'data/curricula/{arm}'], check=True)


## Persist data and train
Use a new experiment name for each run. Rerunning the same configuration needs `--resume latest` and an existing periodic checkpoint. A final adapter alone is not an optimizer checkpoint.

In [ ]:
from pathlib import Path
import json, shutil
project = Path('/content/drive/MyDrive/contrastforge_experiment_01')
project.mkdir(parents=True, exist_ok=True)
shutil.copytree('data', project/'data', dirs_exist_ok=True)
cfg = json.loads(Path('configs/contrast_sft_colab.json').read_text())
cfg['train_data'] = str(project/'data/curricula/mined/sft_train.jsonl')
cfg['validation_data'] = str(project/'data/curricula/mined/sft_validation.jsonl')
cfg['output_dir'] = str(project/'sft_adapter')
Path('configs/colab_active.json').write_text(json.dumps(cfg, indent=2))
subprocess.run([sys.executable, '-m', 'contrastforge.cli', 'train', '--config', 'configs/colab_active.json'], check=True)


## Matched held-out evaluation
Evaluate the base and the adapter with identical prompts, budgets and quantization. These results measure this generated benchmark; add external retention tasks.

In [ ]:
for name, adapter in [('base', None), ('sft', str(project/'sft_adapter'))]:
    command = [sys.executable, '-m', 'contrastforge.cli', 'collect', '--inputs', 'data/contrast/test.jsonl', '--out', str(project/f'{name}_test.jsonl'), '--model', 'models/Qwen3-8B', '--quantize', '--budgets', '256', '512', '1024']
    if adapter: command += ['--adapter', adapter]
    subprocess.run(command, check=True)
    subprocess.run([sys.executable, '-m', 'contrastforge.cli', 'evaluate', '--inputs', 'data/contrast/test.jsonl', '--rollouts', str(project/f'{name}_test.jsonl'), '--out', str(project/f'{name}_report.json')], check=True)
subprocess.run([sys.executable, '-m', 'contrastforge.cli', 'compare', '--reports', str(project/'base_report.json'), str(project/'sft_report.json'), '--out', str(project/'comparison.json')], check=True)
print((project/'comparison.json').read_text())


Optional DPO, teacher assistance, LIC and vision workflows are in `docs/CONTRAST_RUNBOOK.md`. Do not use `--allow-synthetic` for real experiments.